In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import re

In [0]:
schema = [
    "id",
    "recordDate",
    "temperature"
]

data = [
    (1, "2015-01-01", 10),
    (2, "2015-01-02", 25),
    (3, "2015-01-03", 20),
    (4, "2015-01-04", 30)
]

weather = spark.createDataFrame(data=data, schema=schema)
# display(weather)
WindowSpec = Window.orderBy(col('recordDate'))  # No partition defined, all data moved to a single partition
weather1 = weather.withColumn('prev_day_temparature', lag(col('temperature'), 1).over(WindowSpec))\
    .withColumn('prev_recordDate', lag(col('recordDate'), 1).over(WindowSpec))
# display(weather1)
weather2 = weather1.filter(col('prev_day_temparature').isNotNull())
weather3 = weather2.filter(col('temperature') > col('prev_day_temparature'))\
    .filter(date_diff(col('recordDate'), col('prev_recordDate')) == 1)\
        .select('id')
display(weather3)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id
2
4


In [0]:
trips_schema = [
    "id",
    "client_id",
    "driver_id",
    "city_id",
    "status",
    "request_at"
]

trips_data = [
    (1, 1, 10, 1, "completed", "2013-10-01"),
    (2, 2, 11, 1, "cancelled_by_driver", "2013-10-01"),
    (3, 3, 12, 6, "completed", "2013-10-01"),
    (4, 4, 13, 6, "cancelled_by_client", "2013-10-01"),
    (5, 1, 10, 1, "completed", "2013-10-02"),
    (6, 2, 11, 6, "completed", "2013-10-02"),
    (7, 3, 12, 6, "completed", "2013-10-02"),
    (8, 2, 12, 12, "completed", "2013-10-03"),
    (9, 3, 10, 12, "completed", "2013-10-03"),
    (10, 4, 13, 12, "cancelled_by_driver", "2013-10-03")
]

users_schema = [
    "users_id",
    "banned",
    "role"
]

users_data = [
    (1, "No", "client"),
    (2, "Yes", "client"),
    (3, "No", "client"),
    (4, "No", "client"),
    (10, "No", "driver"),
    (11, "No", "driver"),
    (12, "No", "driver"),
    (13, "No", "driver")
]

trips_df = spark.createDataFrame(trips_data, trips_schema)
users_df = spark.createDataFrame(users_data, users_schema)

# trips_df.show()
# users_df.show()

client_users_df = trips_df.join(users_df, trips_df['client_id'] == users_df['users_id'], 'inner')\
                        .filter(col('banned') == 'No')
driver_users_df = trips_df.join(users_df, trips_df['driver_id'] == users_df['users_id'], 'inner')\
                        .filter(col('banned') == 'No')

trips_users_df = client_users_df.unionByName(driver_users_df)
# display(trips_users_df)
pre_result_df = trips_users_df.groupBy(col('request_at')).agg(count(lit(1)).alias('total_trips'), sum(when(col('status') != 'completed', 1).otherwise(0)).alias('cancelled_trips'))
# display(pre_result_df)
result_df = pre_result_df.withColumn('trips', col('total_trips') - col('cancelled_trips'))\
                        .withColumn('Cancellation_Rate', round(col('cancelled_trips') / col('total_trips'), 2))
display(result_df)

request_at,total_trips,cancelled_trips,trips,Cancellation_Rate
2013-10-01,7,3,4,0.43
2013-10-02,5,0,5,0.0
2013-10-03,5,2,3,0.4


In [0]:
schema = [
    "player_id",
    "device_id",
    "event_date",
    "games_played"
]

data = [
    (1, 2, "2016-03-01", 5),
    (1, 2, "2016-05-02", 6),
    (2, 3, "2017-06-25", 1),
    (3, 1, "2016-03-02", 0),
    (3, 4, "2018-07-03", 5)
]

activity = spark.createDataFrame(data, schema)
# activity.show()
WindowSpec = Window.partitionBy('player_id').orderBy('event_date')
pre_results = activity.withColumn('_rank', rank().over(WindowSpec))
results = pre_results.filter(col('_rank') == 1).drop('_rank')\
    .select('player_id', 'event_date')
display(results)

player_id,event_date
1,2016-03-01
2,2017-06-25
3,2016-03-02


In [0]:
data = [
    (1, 2, "2016-03-01", 5),
    (1, 2, "2016-03-02", 6),
    (2, 3, "2017-06-25", 1),
    (3, 1, "2016-03-02", 0),
    (3, 4, "2018-07-03", 5)
]

activity = spark.createDataFrame(data, schema)

streaks = activity.withColumn('prev_event_date', lag('event_date', 1).over(WindowSpec))\
    .filter(col('prev_event_date').isNotNull())\
        .withColumn('gaps', date_diff('event_date', 'prev_event_date'))\
            .filter(col('games_played') > 0)\
                .filter(col('gaps') == 1)

activity_1 = activity.groupBy().agg(countDistinct(col('player_id')).alias('total_users'))
activity_2 = streaks.groupBy().agg(count(lit(1)).alias('streaks_users'))

result = activity_2.crossJoin(activity_1)\
    .withColumn('fraction', round(col('streaks_users') / col('total_users'), 2))\
    .select('fraction')

display(result)

fraction
0.33


In [0]:
employees_data = [
    (101, "John", "A", None),
    (102, "Dan", "A", 101),
    (103, "James", "A", 101),
    (104, "Amy", "A", 101),
    (105, "Anne", "A", 101),
    (106, "Ron", "B", 101)
]

columns = ["id", "name", "department", "managerId"]

employees = spark.createDataFrame(employees_data, columns)
# display(employees)
employees = employees.alias('employees')
managers = employees.alias('managers')
managers_with_at_least_5reportees = employees.join(managers, col('employees.managerId') == col('managers.id'), 'inner')\
    .groupBy(col('employees.managerId')).agg(count(lit(1)).alias('total_reportees'))\
        .where(col('total_reportees') >= 5)\
            .select('managerId')
display(managers_with_at_least_5reportees)

managerId
101


In [0]:
columns = ["pid", "tiv_2015", "tiv_2016", "lat", "lon"]

insrnce = [
    (1, 10, 5, 10, 10),
    (2, 20, 20, 20, 20),
    (3, 10, 30, 20, 20),
    (4, 10, 40, 40, 40)
]

insurance = spark.createDataFrame(insrnce, columns)
# display(insurance)

insurance_alias = insurance.alias('ins')
asked = insurance.groupBy(col('tiv_2015')).agg(count(lit(1)).alias('total'))\
    .where(col('total') > 1).select('tiv_2015')

not_same_city = insurance.groupBy(col('lat'), col('lon')).agg(count(lit(1)).alias('total'))\
    .where(col('total') == 1).select('lat', 'lon')

insurance = insurance_alias.join(asked, col('ins.tiv_2015') == asked['tiv_2015'], 'inner')\
    .join(not_same_city, (col('ins.lat') == not_same_city['lat']) & (col('ins.lon') == not_same_city['lon']), 'inner')\
    .select(col('ins.pid'), col('ins.tiv_2015'), col('ins.tiv_2016'), col('ins.lat'), col('ins.lon'))
    
insurance = insurance.groupBy().agg(sum(col('tiv_2016')).alias('tiv_2016'))
insurance = insurance.withColumn('tiv_2016', round(col('tiv_2016'), 2))
insurance = insurance.select('tiv_2016')
display(insurance)

tiv_2016
45


In [0]:
employee_schema = ["empId", "name", "supervisor", "salary"]

employee_data = [
    (3, "Brad", None, 4000),
    (1, "John", 3, 1000),
    (2, "Dan", 3, 2000),
    (4, "Thomas", 3, 4000)
]

bonus_schema = ["empId", "bonus"]

bonus_data = [
    (2, 500),
    (4, 2000)
]

employees = spark.createDataFrame(employee_data, employee_schema)
bonus = spark.createDataFrame(bonus_data, bonus_schema)

result = employees.join(bonus, employees['empId'] == bonus['empId'], 'left')\
    .filter((col('bonus').isNull()) | (col('bonus') < 1000))\
            .select('name', 'bonus')

display(result)

name,bonus
Brad,null
John,null
Dan,500


In [0]:
my_numbers_schema = ["num"]

my_numbers_data = [
    (8,),
    (8,),
    (3,),
    (3,),
    (1,),
    (4,),
    (5,),
    (6,)
]

my_numbers = spark.createDataFrame(my_numbers_data, my_numbers_schema)
# my_numbers.show()
single_nonRepeatingNumber = my_numbers.groupBy(col('num')).agg(count(lit(1)).alias('repetation'))\
    .filter(col('repetation') == 1)\
        .groupBy().agg(max('num').alias('num'))\
            .select('num')

display(single_nonRepeatingNumber)

num
6


In [0]:
cinema_schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('movie', StringType(), True),
    StructField('description', StringType(), True),
    StructField('rating', FloatType(), True)
])

cinema_data = [
    (1, "War", "great 3D", 8.9),
    (2, "Science", "fiction", 8.5),
    (3, "irish", "boring", 6.2),
    (4, "Ice song", "Fantacy", 8.6),
    (5, "House card", "Interesting", 9.1)
]

cinema = spark.createDataFrame(data=cinema_data, schema=cinema_schema)
asked = cinema.filter(col('id') %2 == 1)\
    .filter(col('description') != 'boring')\
        .orderBy(desc("rating"))
display(asked)

id,movie,description,rating
5,House card,Interesting,9.1
1,War,great 3D,8.9


In [0]:
orders_schema = ["order_number", "customer_number"]

orders_data = [
    (1, 1),
    (2, 2),
    (3, 3),
    (4, 3)
]

orders = spark.createDataFrame(orders_data, orders_schema)
# display(orders)

result = orders.groupBy('customer_number').agg(count('order_number').alias('total_orders'))\
    .orderBy(desc('total_orders'))\
        .limit(1)\
            .select('customer_number')
display(result)

customer_number
3


In [0]:
courses_schema = ["student", "class"]

courses_data = [
    ("A", "Math"),
    ("B", "English"),
    ("C", "Math"),
    ("D", "Biology"),
    ("E", "Math"),
    ("F", "Computer"),
    ("G", "Math"),
    ("H", "Math"),
    ("I", "Math")
]

courses = spark.createDataFrame(data=courses_data, schema=courses_schema)

result = courses.groupBy('class').agg(count(lit(1)).alias('cnt'))\
    .filter(col('cnt') >= 5)
display(result)

class,cnt
Math,6


In [0]:
request_accepted_schema = ["requester_id", "accepter_id", "accept_date"]

request_accepted_data = [
    (1, 2, "2016/06/03"),
    (1, 3, "2016/06/08"),
    (2, 3, "2016/06/08"),
    (3, 4, "2016/06/09")
]

request_accepted = spark.createDataFrame(request_accepted_data, request_accepted_schema)
# display(request_accepted)
df1 = request_accepted.select(col('requester_id').alias('id'))
df2 = request_accepted.select(col('accepter_id').alias('id'))

result = df1.union(df2)\
    .groupBy('id').agg(count(lit(1)).alias('num'))\
        .orderBy(desc('num'))\
            .limit(1)
display(result)

id,num
3,3


In [0]:
stadium_schema = ["id", "visit_date", "people"]

stadium_data = [
    (1, "2017-01-01", 10),
    (2, "2017-01-02", 109),
    (3, "2017-01-03", 150),
    (4, "2017-01-04", 99),
    (5, "2017-01-05", 145),
    (6, "2017-01-06", 1455),
    (7, "2017-01-07", 199),
    (8, "2017-01-09", 188)
]

stadium = spark.createDataFrame(stadium_data, stadium_schema)
display(stadium)
stadium.createOrReplaceTempView("stadium")
result_sql = spark.sql("""
                WITH qualified AS (
                        SELECT
                            id,
                            visit_date,
                            people,
                            id - ROW_NUMBER() OVER (ORDER BY id) AS grp
                        FROM Stadium
                        WHERE people >= 100
                    ),
                    valid_groups AS (
                        SELECT grp
                        FROM qualified
                        GROUP BY grp
                        HAVING COUNT(*) >= 3
                    )
                SELECT
                    q.id,
                    q.visit_date,
                    q.people
                FROM qualified q
                JOIN valid_groups vg
                    ON q.grp = vg.grp
                ORDER BY q.visit_date ASC;
            """)
display(result_sql)

qualified = stadium.select(col('id'), col('visit_date'), col('people'))\
    .filter(col('people') >= 100)\
        .withColumn('grp', col('id') - row_number().over(Window.orderBy(col('id'))))

result = qualified.groupBy('grp').agg(count('*').alias('cnt'))\
    .filter(col('cnt') >= 3)\
        .join(qualified, 'grp')\
            .select(col('id'), col('visit_date'), col('people'))
display(result)

id,visit_date,people
1,2017-01-01,10
2,2017-01-02,109
3,2017-01-03,150
4,2017-01-04,99
5,2017-01-05,145
6,2017-01-06,1455
7,2017-01-07,199
8,2017-01-09,188


id,visit_date,people
5,2017-01-05,145
6,2017-01-06,1455
7,2017-01-07,199
8,2017-01-09,188


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id,visit_date,people
5,2017-01-05,145
6,2017-01-06,1455
7,2017-01-07,199
8,2017-01-09,188


In [0]:
salesperson_schema = ["sales_id", "name", "salary", "commission_rate", "hire_date"]

salesperson_data = [
    (1, "John", 100000, 6, "4/1/2006"),
    (2, "Amy", 12000, 5, "5/1/2010"),
    (3, "Mark", 65000, 12, "12/25/2008"),
    (4, "Pam", 25000, 25, "1/1/2005"),
    (5, "Alex", 5000, 10, "2/3/2007")
]

company_schema = ["com_id", "name", "city"]

company_data = [
    (1, "RED", "Boston"),
    (2, "ORANGE", "New York"),
    (3, "YELLOW", "Boston"),
    (4, "GREEN", "Austin")
]

orders_schema = ["order_id", "order_date", "com_id", "sales_id", "amount"]

orders_data = [
    (1, "1/1/2014", 3, 4, 10000),
    (2, "2/1/2014", 4, 5, 5000),
    (3, "3/1/2014", 1, 1, 50000),
    (4, "4/1/2014", 1, 4, 25000)
]

salesperson = spark.createDataFrame(salesperson_data, salesperson_schema)
company = spark.createDataFrame(company_data, company_schema)
orders = spark.createDataFrame(orders_data, orders_schema)

# Find salespersons who have orders with RED company
salespersons_with_red = orders.join(company, orders['com_id'] == company['com_id'])\
    .filter(col('name') == 'RED')\
    .select('sales_id').distinct()

# Get all salespersons EXCEPT those who have orders with RED (using left_anti join)
result = salesperson.join(salespersons_with_red, 'sales_id', 'left_anti')\
    .select('name')
display(result)

name
Amy
Mark
Alex


In [0]:
triangle_schema = ["x", "y", "z"]

triangle_data = [
    (13, 15, 30),
    (10, 20, 15)
]

triangle = spark.createDataFrame(triangle_data, triangle_schema)
# display(triangle)
result = triangle.withColumn('triangle', 
    when((col('x') + col('y') > col('z')) & (col('x') + col('z') > col('y')) & (col('y') + col('z') > col('x')), 'Yes')
    .otherwise('No'))
display(result)

x,y,z,triangle
13,15,30,No
10,20,15,Yes
